<p align="center">
  <img src="./Figures/Label_Instruction.png"
       width="900">
</p>


# Summer School Delegate Guide
## GROMACS 2026.3 on WSL 2, with VMD and Avogadro

**Audience:** summer-school delegates using Windows 10/11 computers.  
**Simulation setup:** CPU-only GROMACS; no CUDA or NVIDIA GPU is required.  
**Recommended layout:** GROMACS runs inside Ubuntu on WSL 2; VMD and Avogadro run as native Windows applications.

This gives Linux compatibility for GROMACS and reliable Windows graphics for molecular visualization. Windows applications can open files stored inside WSL through `\\wsl$` or by running `explorer.exe .` in WSL.

### Documented versions

- GROMACS 2026.3
- Ubuntu 24.04 on WSL 2 (recommended for new delegates)
- Ubuntu 20.04 fallback instructions included
- VMD 1.9.4 Windows 64-bit build from the official portal
- Avogadro 2.0.0 Windows 64-bit release

> Instructors should trial the complete installation on the same Windows and Ubuntu versions delegates will use.


## 1. Know where each instruction runs

| Label | Where to run it | Typical prompt |
|---|---|---|
| **PowerShell (Administrator)** | Windows Start → PowerShell → Run as administrator | `PS C:\\WINDOWS\\System32>` |
| **WSL terminal** | Ubuntu application or `wsl` from PowerShell | `delegate@computer:~$` |
| **Notebook Bash cell** | Jupyter started inside WSL | begins with `%%bash` |
| **Windows installer** | File Explorer / web browser | graphical installer |

Do not enter PowerShell commands in Ubuntu or Linux commands in PowerShell. Steps containing `sudo` are shown as terminal blocks because password prompts are more reliable there.


## 2. Delegate checklist before installation

Each computer should have:

- 64-bit Windows 10 build 19044+ or Windows 11
- administrator permission for initial installation
- at least 15 GB free disk space
- at least 8 GB RAM (16 GB or more preferred)
- a stable internet connection
- virtualization enabled in firmware

No NVIDIA GPU or CUDA Toolkit is needed.


## 3. Install or verify WSL 2

Open **PowerShell as Administrator**. For a new installation:

```powershell
wsl --install -d Ubuntu-24.04
```

Restart Windows if requested. Verify an existing installation with:

```powershell
wsl --list --verbose
```

Ubuntu must show `VERSION 2`. Update and restart WSL:

```powershell
wsl --update
wsl --shutdown
```

Launch Ubuntu from Start. On first launch, create a Linux username and password. The password is not displayed while typing; this is normal.


## 4. Install Ubuntu prerequisites

Run in the **WSL terminal**:

```bash
sudo apt update
sudo apt install -y build-essential cmake wget tar ca-certificates pkg-config
```

Ubuntu 24.04 normally supplies suitable GCC and CMake versions.


In [ ]:
%%bash
set -euo pipefail
grep PRETTY_NAME /etc/os-release
gcc --version | head -n 1
g++ --version | head -n 1
cmake --version | head -n 1


### Required versions

- CMake 3.28 or newer
- GCC/G++ 11 or newer

If both are satisfied, continue to Section 5.

### Ubuntu 20.04 fallback

Ubuntu 20.04 defaults are too old. Run in its WSL terminal:

```bash
sudo apt install -y software-properties-common
sudo add-apt-repository -y ppa:ubuntu-toolchain-r/test
sudo apt update
sudo apt install -y gcc-11 g++-11
conda install -y -c conda-forge cmake=3.31
hash -r
```

This assumes Miniforge/Conda is present. Otherwise use Ubuntu 24.04 or ask an instructor for the prepared Ubuntu 20.04 setup.


## 5. Download GROMACS 2026.3

Keep source, build, and simulations in the Linux home directory for best WSL performance. Avoid building under `/mnt/c`.


In [ ]:
%%bash
set -euo pipefail
cd "$HOME"
if [ ! -f gromacs-2026.3.tar.gz ]; then wget https://ftp.gromacs.org/gromacs/gromacs-2026.3.tar.gz; fi
if [ ! -d gromacs-2026.3 ]; then tar xzf gromacs-2026.3.tar.gz; fi
ls -ld "$HOME/gromacs-2026.3"


## 6. Configure a CPU-only build

- `GMX_GPU=OFF`: no CUDA/GPU build
- `GMX_MPI=OFF`: no external multi-node MPI; local thread-MPI remains available
- `GMX_DOUBLE=OFF`: normal single precision
- `GMX_BUILD_OWN_FFTW=ON`: builds a suitable FFTW library
- `REGRESSIONTEST_DOWNLOAD=ON`: downloads official tests

### Ubuntu 24.04 command


In [ ]:
%%bash
set -euo pipefail
mkdir -p "$HOME/gromacs-2026.3/build"
cd "$HOME/gromacs-2026.3/build"
cmake --fresh .. -DCMAKE_INSTALL_PREFIX=/usr/local/gromacs-2026.3 -DGMX_BUILD_OWN_FFTW=ON -DREGRESSIONTEST_DOWNLOAD=ON -DCMAKE_C_COMPILER=/usr/bin/gcc -DCMAKE_CXX_COMPILER=/usr/bin/g++ -DGMX_GPU=OFF -DGMX_MPI=OFF -DGMX_DOUBLE=OFF -DCMAKE_BUILD_TYPE=Release


### Ubuntu 20.04 command

Use this instead after installing GCC 11 and CMake 3.31:

```bash
cd ~/gromacs-2026.3/build
cmake --fresh .. -DCMAKE_INSTALL_PREFIX=/usr/local/gromacs-2026.3 -DGMX_BUILD_OWN_FFTW=ON -DREGRESSIONTEST_DOWNLOAD=ON -DCMAKE_C_COMPILER=/usr/bin/gcc-11 -DCMAKE_CXX_COMPILER=/usr/bin/g++-11 -DGMX_GPU=OFF -DGMX_MPI=OFF -DGMX_DOUBLE=OFF -DCMAKE_BUILD_TYPE=Release
```


## 7. Compile and test

Compilation may take several minutes.


In [ ]:
%%bash
set -euo pipefail
cd "$HOME/gromacs-2026.3/build"
make -j"$(nproc)"


In [ ]:
%%bash
set -euo pipefail
cd "$HOME/gromacs-2026.3/build"
make check


## 8. Install and activate GROMACS

After tests pass, run in WSL:

```bash
cd ~/gromacs-2026.3/build
sudo make install
source /usr/local/gromacs-2026.3/bin/GMXRC
grep -qxF 'source /usr/local/gromacs-2026.3/bin/GMXRC' ~/.bashrc || echo 'source /usr/local/gromacs-2026.3/bin/GMXRC' >> ~/.bashrc
source ~/.bashrc
gmx --version
```

Use the versioned path shown above, not `/usr/local/gromacs/bin/GMXRC`.


In [ ]:
%%bash
set -eo pipefail
source /usr/local/gromacs-2026.3/bin/GMXRC
gmx --version

Confirm GROMACS 2026.3, GPU support disabled, OpenMP enabled, and an automatically selected SIMD level.


## 9. Create course working folders


In [ ]:
%%bash
set -euo pipefail
mkdir -p "$HOME/summer-school/structures" "$HOME/summer-school/topology" "$HOME/summer-school/mdp" "$HOME/summer-school/runs" "$HOME/summer-school/analysis"
find "$HOME/summer-school" -maxdepth 1 -type d -print


Open the WSL course folder in Windows File Explorer with:

```bash
cd ~/summer-school
explorer.exe .
```


## 10. Install VMD on Windows (recommended)

VMD is distributed under a University of Illinois license. Each delegate must register, accept the license, and download their own copy; organizers should not redistribute the installer.

1. Open https://www.ks.uiuc.edu/Development/Download/download.cgi?PackageName=VMD
2. Register or sign in.
3. Under **Version 1.9.4**, choose the **Windows 64-bit** build specified by the instructors.
4. Accept the license and download the installer.
5. Run it with the default options.
6. Launch VMD from Start.

A VMD build name may mention CUDA, but CUDA is not required for basic visualization. This does not change the CPU-only GROMACS build.

### Test

Run `explorer.exe .` in a WSL course directory. In VMD choose **File → New Molecule**, open a course `.pdb` or `.gro`, and confirm that it displays.


## 11. Install Avogadro on Windows (recommended)

Avogadro 2.0.0 is the current official release.

1. Open https://avogadro.cc/install/
2. Under **Windows**, click **Download Installer**.
3. Run `Avogadro2-2.0.0-win64.exe`.
4. Accept the default options and launch Avogadro 2.

Documented installer: https://github.com/OpenChemistry/avogadrolibs/releases/download/2.0.0/Avogadro2-2.0.0-win64.exe

### Test

Choose **File → Open**, open a course `.xyz`, `.mol`, or `.pdb`, and confirm that the molecule appears and rotates.


## 12. Optional: Avogadro inside WSL using WSLg

Use this only if instructors require the Linux app. Native Windows is the default. Update WSL from Administrator PowerShell with `wsl --update` and `wsl --shutdown`, then run in WSL:

```bash
mkdir -p ~/.local/bin
wget -O ~/.local/bin/Avogadro2.AppImage https://github.com/OpenChemistry/avogadrolibs/releases/latest/download/Avogadro2-x86_64.AppImage
chmod +x ~/.local/bin/Avogadro2.AppImage
~/.local/bin/Avogadro2.AppImage --appimage-extract-and-run
```

If no window appears, use the native Windows version. Linux VMD under WSL is not the default delegate route because its licensed archive, OpenGL requirements, and build-specific dependencies make a uniform classroom setup less reliable.


## 13. Working across WSL and Windows

1. Run GROMACS and store simulations in `~/summer-school`.
2. In a project directory, run `explorer.exe .`.
3. Open files in native Windows VMD or Avogadro from that window.
4. Use `explorer.exe .` or `\\wsl$`; never edit WSL's hidden virtual-disk files directly.

| File | Purpose | Application |
|---|---|---|
| `.pdb`, `.gro` | structure | VMD or Avogadro |
| `.xtc`, `.trr` | trajectory | VMD |
| `.top`, `.itp` | topology | text editor |
| `.mdp` | simulation parameters | text editor |
| `.xyz`, `.mol`, `.sdf` | small molecule | Avogadro |


## 14. Troubleshooting

### CMake 3.28+ required
Ubuntu 20.04 is using old CMake. Use the fallback or Ubuntu 24.04.

### `/usr/bin/gcc-11` missing
Install both `gcc-11` and `g++-11`, then verify their paths.

### CMake asks for `nvcc`
A CUDA command or stale cache was used. Run the exact `cmake --fresh` command with `GMX_GPU=OFF`.

### `gmx: command not found`
Run `source /usr/local/gromacs-2026.3/bin/GMXRC`.

### `GMXRC` not found
Use `/usr/local/gromacs-2026.3/bin/GMXRC`, not `/usr/local/gromacs/bin/GMXRC`.

### VMD cannot be downloaded
Each delegate must register and accept the official license.

### Windows apps cannot see WSL files
Run `explorer.exe .` from the WSL project directory.

### WSL GUI app does not open
Run `wsl --update` and `wsl --shutdown` in Administrator PowerShell, or use the recommended native Windows app.


## 15. Delegate completion checklist

- [ ] WSL reports version 2
- [ ] CMake is 3.28+
- [ ] GCC is 11+
- [ ] `make check` passes
- [ ] `gmx --version` reports 2026.3 and GPU disabled
- [ ] `~/summer-school` exists
- [ ] VMD opens a test structure
- [ ] Avogadro opens a test molecule
- [ ] `explorer.exe .` opens the current WSL folder


## 16. Instructor pre-flight checklist

- Standardize on Ubuntu 24.04 where possible.
- Test the build on the slowest expected computer.
- Specify the exact VMD build delegates should select.
- Provide small `.pdb`, `.gro`, `.xtc`, and `.xyz` test files.
- Confirm venue internet permits all official download sites.
- Ask delegates to install before the first practical.
- Prepare several configured loan computers.
- Nominate a support contact for screenshots and logs.


## 17. Official references

- GROMACS guide: https://manual.gromacs.org/current/install-guide/index.html
- GROMACS archive: https://ftp.gromacs.org/gromacs/
- Microsoft WSL: https://learn.microsoft.com/windows/wsl/
- WSL GUI apps: https://learn.microsoft.com/windows/wsl/tutorials/gui-apps
- WSL file systems: https://learn.microsoft.com/windows/wsl/filesystems
- VMD: https://www.ks.uiuc.edu/Research/vmd/
- VMD download: https://www.ks.uiuc.edu/Development/Download/download.cgi?PackageName=VMD
- Avogadro install: https://avogadro.cc/install/
- Avogadro docs: https://avogadro.cc/docs/
